# Artificial Intelligence in Health Care - Exercise 2
## Convolutional Neural Networks for Image Classification

Mitglieder: Elbischger Selina, Krauss Marco
Jahrgang: EHT25

Das Ziel der Übung ist es ein CNN (Convolusional neural Network) zu trainieren um damit verschiedene histologische Bilder von Darmgewebe der passenden Kategorie (Darmschleimhaut, Tumorgewebe, Muskel, etc. ) zuordnen zu können.

In [ ]:
#Imports and Configs
import os
os.environ["KERAS_BACKEND"] = "torch"

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")
SEED = 42
keras.utils.set_random_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})
print("keras", keras.__version__, "| backend", keras.backend.backend())

## Datenimport

In [ ]:
import numpy as np
data = np.load("C:\\Users\\Marco\\Desktop\\Repos\\KIGW\\data\\path_train.npz")
print(list(data.keys()))

x_train, y_train = data["x_train"], data["y_train"]
x_val,   y_val   = data["x_val"],   data["y_val"]
x_test,  y_test  = data["x_test"],  data["y_test"]

#Kontrolle mit Angabenzettel Anzahl der Daten
print(x_train.shape, x_val.shape, x_test.shape)

### 1 Data Pre-Processing and Partitioning [2P]:


Make yourself familiar with the dataset by plotting one example image per class, arranged in
a figure of e.g. two rows, including class names and labels. Prepare class labels for the use
with a neural network and multi-class classification. The images are given in a range of [0, 255];
normalise them to [0, 1]. Report the class distribution of the training set. You will need it when
you interpret the confusion matrix in tasks 2 and 3.

#### Labels in 1D Array, Klassennamen festlegen 

In [ ]:
#In 1D Arrays umwandeln, da Keras sonst Probleme hat
y_train = y_train.ravel(); y_val = y_val.ravel(); y_test = y_test.ravel()  

#Klassen aus Angabe übernommen, da die Klassenbezeichnungen nicht importiert wird wie bei medmnist
CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]


#### Beispielbild pro Klasse

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(9, 4.6))
for i, ax in enumerate(axes.ravel()):
    #if bedingung sonst gibt es einen index error, da es nur 9 Klassen gibt und wir 10 subplots haben
    if i < len(CLASSES):
        ax.imshow(x_train[np.where(y_train == i)[0][0]])
        ax.set_title(f"{i} · {CLASSES[i]}", fontsize=8)
        ax.axis("off")
fig.suptitle("BloodMNIST — one example per class", fontsize=10)
plt.tight_layout(); plt.show()


#### Klassenverteilung y_train

In [ ]:
#Ausgabe der Datensatzgrößen und Klassenverteilung
print("train / val / test:", len(y_train), len(y_val), len(y_test))
print("image shape       :", x_train.shape[1:])
print("classes           :", len(CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {i}  {c:<28} {(y_train==i).sum():>6} {100*(y_train==i).mean():>5.1f}%")
print()
print("majority-class baseline accuracy: %.3f" % (np.bincount(y_test).max()/len(y_test)))

#### Bilder auf [0, 1] normalisieren

In [ ]:
#auf 0-1 normalisieren wie bei NB3
x_train = x_train.astype("float32") / 255.0
x_val   = x_val.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

#Check min und max wert, sollte 0 und 1 sein
print(x_train.min(), x_train.max())   

#### Labels One-Hot-kodieren

In [ ]:
#pklassenlabels für One-Hot-Encoding vorbereiten
num_classes = len(CLASSES)

y_train_oh = keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = keras.utils.to_categorical(y_val,   num_classes)
y_test_oh  = keras.utils.to_categorical(y_test,  num_classes)

#checken, ob die One-Hot-Encoding rihtig funktioniert hat, erste 3 labels ausgeben und die Form des Arrays
print(y_train[:3]); 
print(y_train_oh[:3]); 
print(y_train_oh.shape)

#### 2 Fully-Connected Neural Network [3P]:

First implement a dense neural network with 3 hidden layers (1024, 512 and 128 units) with
relu activation and an output layer using softmax activation. For training, use the categorical
cross-entropy loss, early stopping on the validation set V, and e.g. the Adam optimizer with an
initial learning rate of α = 0.001. Consider the use of Dropout layers.
In contrast to Exercise 1, the input data here are images. To pass them to a dense network,
flatten them in the input layer using keras.layers.Flatten. Plot the learning curves for the training and validation set (loss and accuracy). Evaluate the
classification performance (accuracy, confusion matrix, AUROC) on the test set. Report the
number of trainable parameters to compare it against the CNN.

In [ ]:
def train_model(model, epochs=5, tag=""):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])
    
    es = keras.callbacks.EarlyStopping(monitor="val_acc", mode="max",
                                       patience=5, restore_best_weights=True)
    
    h = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=epochs,
                  batch_size=128, verbose=0, callbacks=[es])
    
    logits = model.predict(x_test, verbose=0)
    prob = keras.ops.softmax(logits).numpy() if hasattr(keras, "ops") else logits
    pred = prob.argmax(1) # argmax along the class dimension for each sample
    
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, prob, multi_class="ovr", average="macro")
    n_par = model.count_params()
    print(f"{tag:<22} test accuracy {acc:.4f}   macro AUROC {auc:.4f}   "
          f"parameters {n_par:,}   epochs {len(h.history['loss'])}")
    return dict(tag=tag, model=model, hist=h, acc=acc, auc=auc, params=n_par,
                prob=prob, pred=pred)


keras.utils.set_random_seed(SEED)
dense = keras.Sequential([
    layers.Input(shape=(28, 28, 3)),
    layers.Flatten(),                      # 28*28*3 = 2352 numbers, geometry gone
    layers.Dense(1024, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(CLASSES)),            # logits
], name="dense")

res_dense = train_model(dense, tag="dense network")